<a href="https://colab.research.google.com/github/SerraGuney/Optimized-Transfer-Learning-for-Multi-Class-Plant-Disease-Classification/blob/main/Optimized_Transfer_Learning_for_Multi_Class_Plant_Disease_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Veri setini colab dosyasına indirme

# Bu çalışmanın amacı, PlantVillage veri setindeki çok sınıflı yaprak görüntülerini kullanarak, 38 farklı sınıfa ait bitki hastalıklarını derin öğrenme ve transfer learning yöntemleri ile doğru şekilde sınıflandırmaktır.

Mobilenet ile trasfer learning gerçekleştircez.çünkü daha az karmaşık ve kullanması daha kolaydır.

**1. Veri Yükleme ve data augmentation**

In [ ]:
from google.colab import userdata
import os

# 1. Şifreleri gizli bölgeden (Secrets) çek
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

# 2. Veriyi indir ve aç (Hızlı disk üzerinde çalışır)
if not os.path.exists('/content/data/color'):
    !kaggle datasets download -d abdallahalidev/plantvillage-dataset
    !unzip -q plantvillage-dataset.zip "color/*" -d /content/data
    !rm plantvillage-dataset.zip # Kalabalık yapmasın, zip'i sil
    print(" Veri seti hazır!")
else:
    print(" Veriler zaten klasörde mevcut.")

In [ ]:
import os

# Data klasörünün içinde gerçekten ne var?
print("Data içindekiler:", os.listdir('/content/data'))

# Eğer boşsa, bir de ana dizine bakalım, belki yanlış yere açılmıştır
print("Ana dizin içindekiler:", os.listdir('/content'))


** Kütüphane Bilgilendirmeleri**
**torch:** Temel kütüphanedir. GPU destekli çok boyutlu dizileri (tensor) ve matematiksel işlemleri yönetir.

**torch.nn:** Sinir ağı katmanlarını (Convolution, Linear, Dropout vb.) ve kayıp fonksiyonlarını (Loss functions) oluşturmak için kullanılır.

**torch.optim:** Modelin parametrelerini (ağırlıklarını) güncellemek için kullanılan algoritmaları (Adam, SGD vb.) barındırır.

**torchvision.transforms:** Görüntüleri Tensor formatına çevirme, yeniden boyutlandırma ve normalizasyon gibi ön işleme işlemlerini yapar.

**torchvision.datasets:** Dosya sistemindeki görüntüleri otomatik olarak etiketleyip veri kümesi (dataset) nesnesine dönüştürür.

**torchvision.models:** Önceden eğitilmiş (Pre-trained) mimarileri (ResNet50, MobileNetV2 vb.) projeye dahil eder.

**DataLoader: **Veri kümesini bellek yönetimine uygun şekilde küçük gruplara (batch) böler ve karıştırarak (shuffle) modele sunar.

**tqdm:** Döngülerin (eğitim ve test aşamaları) ilerleme durumunu ve süresini konsolda görselleştirir.

**matplotlib.pyplot:**Eğitim sırasındaki hata ve doğruluk oranlarını grafik olarak raporlamak için kullanılır.

**seaborn:** Matplotlib tabanlıdır; karmaşıklık matrisi (confusion matrix) gibi istatistiksel verileri daha okunabilir şekilde görselleştirir.

**sklearn.metrics:** Modelin performansını ölçmek için kesinlik (precision), duyarlılık (recall) ve F1 skoru gibi metrikleri hesaplar.


In [10]:
import torch
import torch.nn as nn # sinir ağı katmanları için
import torch.optim as optim # sinir ağındaki optimizasyon algoritmaları için
import torchvision.transforms as transforms # görüntü dönüşümleri için (data augmentataion)
import torchvision.datasets as datasets # görüntü veri kümeleri için
import torchvision.models as models # önceden eğitilmiş modelleri yüklemek için(mobilenet,...)
from torch.utils.data import DataLoader #batchler oluşturur.
from tqdm import tqdm # eğitim sürecini izlemek için kullancağımız bir ilerleme cubugu
import matplotlib.pyplot as plt # görselleştirme için
import seaborn as sns #confuxion matris için
from sklearn.metrics import confusion_matrix, classification_report #confuxion matris için


**veri dönüşümü**
1.   klasik dönüşümler:normalizasyon, tensor dönüşümü
2.   mobilnet e uygun giris boyutunun ayarlanması
3.   data augmentation





In [ ]:
transform_train=transforms.Compose([
    transforms.Resize((224,224)), #mobile net input size yazmamız lazım.
    transforms.RandomHorizontalFlip(), # görüntüleri yatay çevirerek veri arttırımı.
    transforms.RandomRotation(10), # görüntüyü rastgele 10 dereceye kadar döndürür.
    transforms.ColorJitter(brightness=0.2, contrast=0.2,saturation=0.2, hue=0.1) , # aynı görüntünün renk varyansolarını sağlar.
    transforms.ToTensor(),#goruntüleri tensore(C,H,W) ve piksel değerlerini 0–1 aralığına çevir.
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) #piksel değerlerini normalize eder.
])

**2. Transfer learning tanımlama ve fine tuning ve model kaydetme**

**3. Test ve değerlendirme**